In [ ]:
from typing import TYPE_CHECKING, Any, cast

if TYPE_CHECKING:
    dbutils = cast(Any, None)
    spark = cast(Any, None)


# Databricks notebook source
# Parameters - can be overridden when called from orchestration notebook
dbutils.widgets.text("CATALOG", "sample_synthetic_sap", "Catalog Name")
dbutils.widgets.text("SCHEMA", "sap", "Schema Name")
dbutils.widgets.text("MIN_MATERIALS", "50", "Minimum expected materials")
dbutils.widgets.text("MIN_ORDERS", "1000", "Minimum expected orders")
dbutils.widgets.text("CONFIG_TABLE", "smoke_test_config", "Config Table Name")

CATALOG = dbutils.widgets.get("CATALOG")
SCHEMA = dbutils.widgets.get("SCHEMA")
MIN_MATERIALS = int(dbutils.widgets.get("MIN_MATERIALS"))
MIN_ORDERS = int(dbutils.widgets.get("MIN_ORDERS"))
CONFIG_TABLE = dbutils.widgets.get("CONFIG_TABLE")

print(f"Running Smoke Tests on {CATALOG}.{SCHEMA}")
print(f"Thresholds: Min Materials={MIN_MATERIALS}, Min Orders={MIN_ORDERS}")
print(f"Config Table: {CATALOG}.{SCHEMA}.{CONFIG_TABLE}")

In [ ]:
# Load the Testing Framework
import json

import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql.utils import AnalysisException


class SAPSmokeTester:
    """Core logic for running smoke tests."""
    def __init__(self, spark, config):
        self.spark = spark
        self.config = config
        self.catalog = config['settings']['catalog']
        self.schema = config['settings']['schema']
        self.results = []
        
    def _get_df(self, table_name):
        return self.spark.table(f"{self.catalog}.{self.schema}.{table_name}")

    def log_result(self, table, check_type, status, message, metrics=None):
        self.results.append({
            "Table": table,
            "Check": check_type,
            "Status": status,
            "Message": message,
            "Metrics": metrics
        })

    def run(self):
        print(f"Starting Smoke Test on {self.catalog}.{self.schema}...")
        
        for t_conf in self.config['tables']:
            table_name = t_conf['table']
            try:
                df = self._get_df(table_name)
            except AnalysisException:
                self.log_result(table_name, "Existence", "FAIL", "Table not found")
                continue 

            row_count = df.count()
            
            for check in t_conf['checks']:
                ctype = check['type']
                
                if ctype == 'count':
                    min_r = check.get('min', 1)
                    res = "PASS" if row_count >= min_r else "FAIL"
                    msg = f"Rows: {row_count} (Min: {min_r})"
                    self.log_result(table_name, "Volume", res, msg, row_count)

                elif ctype == 'no_nulls':
                    cols = check['columns']
                    missing = [c for c in cols if c not in df.columns]
                    if missing:
                         self.log_result(table_name, "Schema", "FAIL", f"Missing cols: {missing}")
                         continue
                    
                    null_expr = " OR ".join([f"{c} IS NULL" for c in cols])
                    null_cnt = df.filter(null_expr).count()
                    res = "PASS" if null_cnt == 0 else "FAIL"
                    self.log_result(table_name, "Null Check", res, f"Nulls in {cols}: {null_cnt}", null_cnt)

                elif ctype == 'allowed_values':
                    col, allowed = check['column'], check['values']
                    if col not in df.columns:
                        self.log_result(table_name, "Schema", "FAIL", f"Missing col: {col}")
                        continue
                    
                    bad_cnt = df.filter(~F.col(col).isin(allowed)).count()
                    res = "PASS" if bad_cnt == 0 else "FAIL"
                    self.log_result(table_name, "Validity", res, f"Invalid values in {col}: {bad_cnt}", bad_cnt)

                elif ctype == 'foreign_key':
                    ptable, c_col, p_col = check['parent_table'], check['child_col'], check['parent_col']
                    try:
                        pdf = self._get_df(ptable)
                        orphans = df.join(pdf, df[c_col] == pdf[p_col], "left_anti").count()
                        res = "PASS" if orphans == 0 else "FAIL"
                        self.log_result(table_name, "Integrity", res, f"Orphans in {c_col}: {orphans}", orphans)
                    except Exception as e:
                        self.log_result(table_name, "Integrity", "ERROR", f"Join failed: {e!s}")
        
        return pd.DataFrame(self.results)

In [ ]:
# Load test configuration from Delta table

try:
    config_df = spark.table(f"{CATALOG}.{SCHEMA}.{CONFIG_TABLE}")
    config_json = config_df.select("config_json").collect()[0][0]
    test_config = json.loads(config_json)
    print(f"Loaded config from {CATALOG}.{SCHEMA}.{CONFIG_TABLE}")
    print(f"Found {len(test_config['tables'])} tables to test")
except Exception as e:
    raise RuntimeError(f"Failed to load config: {e!s}. Run the config setup notebook first.") from e

# Override settings from widgets (allows orchestration to control target)
test_config['settings']['catalog'] = CATALOG
test_config['settings']['schema'] = SCHEMA

# Apply dynamic thresholds from widgets to specific tables
for table_conf in test_config['tables']:
    for check in table_conf['checks']:
        if check['type'] == 'count':
            if table_conf['table'] == 'mara':
                check['min'] = MIN_MATERIALS
            elif table_conf['table'] == 'vbak':
                check['min'] = MIN_ORDERS

# Run the tests
tester = SAPSmokeTester(spark, test_config)
df_results = tester.run()

# Display results
display(df_results)

In [ ]:
# Output final status
failed_count = len(df_results[df_results['Status'] == 'FAIL'])
error_count = len(df_results[df_results['Status'] == 'ERROR'])
passed_count = len(df_results[df_results['Status'] == 'PASS'])
total_count = len(df_results)

print(f"\nResults: {passed_count}/{total_count} tests passed")

if failed_count > 0 or error_count > 0:
    status = "FAILED"
    print(f"\nSMOKE TESTS: FAILED ({failed_count} failures, {error_count} errors)")
else:
    status = "PASSED"
    print("\nSMOKE TESTS: ALL PASSED")

# Return status for orchestration notebook
dbutils.notebook.exit(status)